# OPC UA — browse, read, subscribe, write

OPC UA is how modern PLCs, SCADA and MES systems expose their data: an address space of objects, variables and methods, reached over a secure session. IoTCom.Net wraps the OPC Foundation stack; this notebook runs its plant simulator in-process.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.16.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.16.0-preview.1"
#r "nuget: IoTCom.Net.Adapters.OpcUa, 0.16.0-preview.1"

## Start the plant simulator and connect
The client picks the most secure endpoint (Basic256Sha256, SignAndEncrypt); both sides create a self-signed application certificate on first use. `ReadOnly = true` until we decide to write.

In [ ]:
using IoTCom.Net.Adapters.OpcUa;

var probe = new System.Net.Sockets.TcpListener(System.Net.IPAddress.Loopback, 0);
probe.Start();
var uaPort = ((System.Net.IPEndPoint)probe.LocalEndpoint).Port;
probe.Stop();
var plc = OpcUaPlantServer.Create(o => { o.Port = uaPort; o.Interval = TimeSpan.FromMilliseconds(300); });
await plc.StartAsync();
var ua = OpcUaClient.Create(o => { o.UseEndpoint(plc.EndpointUrl); o.AcceptUntrustedCertificates = true; o.ReadOnly = true; });
await ua.ConnectAsync();
Console.WriteLine($"{plc.EndpointUrl} · {ua.SecurityPolicy} / {ua.SecurityMode}");

## Browse and read
Node ids carry a namespace index: `ns=2;s=Plant/Line1/Filler/Speed`.

In [ ]:
async Task Tree(string? nodeId, string indent)
{
    foreach (var n in await ua.BrowseAsync(nodeId))
    {
        if (n.NodeId.StartsWith("i=")) continue;
        var value = n.NodeClass == "Variable" ? " = " + (await ua.ReadAsync(n.NodeId)).Text : "";
        Console.WriteLine($"{indent}{n.DisplayName}{value}");
        if (n.IsContainer) await Tree(n.NodeId, indent + "  ");
    }
}
await Tree(null, "");

## Subscribe
The server samples the monitored items and publishes only changes.

In [ ]:
using var window = new CancellationTokenSource(TimeSpan.FromSeconds(2));
var changes = 0;
try
{
    await foreach (var v in ua.SubscribeAsync([plc.NodeId("Line1/Tank7/Level"), plc.NodeId("Line1/Filler/Speed")], TimeSpan.FromMilliseconds(200), window.Token))
        if (changes++ < 8) Console.WriteLine($"{v.SourceTimestamp?.ToLocalTime():HH:mm:ss.fff} {v.NodeId} = {v.Text}");
}
catch (OperationCanceledException) { }
Console.WriteLine($"{changes} notifications in 2 s");

## Writes are refused until you allow them

In [ ]:
try { await ua.WriteAsync(plc.NodeId("Line1/Filler/Setpoint"), 100); }
catch (IoTCom.Net.ReadOnlyModeException e) { Console.WriteLine("read-only: " + e.Message); }

await using var operatorUa = OpcUaClient.Create(o => { o.UseEndpoint(plc.EndpointUrl); o.AcceptUntrustedCertificates = true; });
await operatorUa.ConnectAsync();
await operatorUa.WriteAsync(plc.NodeId("Line1/Filler/Setpoint"), "100");   // converted to Double
Console.WriteLine($"setpoint now {(await ua.ReadAsync(plc.NodeId("Line1/Filler/Setpoint"))).Text}");
var previous = await operatorUa.CallAsync(plc.NodeId("Line1"), plc.NodeId("Line1/ResetCounter"));
Console.WriteLine($"ResetCounter() returned {previous[0]}");
await ua.DisposeAsync();
await plc.DisposeAsync();

## Going further
`iotcom opcua browse --sim`, `iotcom opcua watch`, the Gallery's *OPC UA tag browser* and the OpcUaBrowser sample. See `docs/en/protocols/opcua.md` for certificates and trust.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*